In [1]:
# ============================================================
# GeoSaúde — Execução via Jupyter Notebook
# ============================================================

import logging
import sys
import time
import traceback
from pathlib import Path

import geosaude
import utils
from utils import get_bbox, calculadora_raster, agregar_resultados, top_cells
from geocnes import geocnes
from report import gerar_relatorio
from dashboard import gerar_dashboard

In [2]:
# ---------------------------------------------------------
# Configuração do logger
# (idêntica ao main.py — pode importar de lá se preferir)
# ---------------------------------------------------------

def configurar_logger(mun: str, uf: str) -> logging.Logger:

    log_dir = Path(f"./data/resultados/{mun}")
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"geosaude_{mun}_{uf}.log"

    logger = logging.getLogger("geosaude")
    logger.setLevel(logging.DEBUG)

    if logger.handlers:
        logger.handlers.clear()

    fmt = logging.Formatter(
        "%(asctime)s  %(levelname)-8s  %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)

    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)

    logger.addHandler(ch)
    logger.addHandler(fh)

    return logger


def executar_etapa(logger, nome, func, *args, **kwargs) -> bool:

    logger.info(f"{'='*55}")
    logger.info(f"INÍCIO: {nome}")
    logger.info(f"{'='*55}")

    t0 = time.time()

    try:
        func(*args, **kwargs)
        logger.info(f"CONCLUÍDO: {nome}  ({time.time() - t0:.1f}s)")
        return True

    except Exception:
        logger.error(f"FALHA: {nome}  ({time.time() - t0:.1f}s)")
        logger.error(traceback.format_exc())
        return False


def criterio_ja_processado(mun: str, nome_arquivo: str) -> bool:
    return Path(
        f"./data/resultados/{mun}/raster/{nome_arquivo}.tif"
    ).exists()


def main(mun, uf, here_api, opentopo_api, forcar_reprocessamento=False):

    logger = configurar_logger(mun, uf)
    logger.info(f"GeoSaúde iniciado → município: {mun} / UF: {uf}")

    t_total = time.time()
    etapas_com_falha = []

    # --- Etapa 0: bbox e CNES ---
    try:
        bbox = get_bbox(mun, uf)
        logger.info(f"Bounding box: {bbox}")
    except Exception:
        logger.critical("Falha ao obter bounding box. Execução interrompida.\n"
                        + traceback.format_exc())
        return  # no notebook usamos return em vez de sys.exit()

    for code_un, descricao in [
        ("02", "Unidades de APS"),
        ("05", "Hospitais"),
        ("73", "UPAs"),
    ]:
        arquivo_cnes = Path(
            f"./data/resultados/{mun}/cnes_{mun}_{uf}_{code_un}.gpkg"
        )
    
        if arquivo_cnes.exists():
            logger.info(f"{descricao} (code_un={code_un}) já existe. Pulando etapa.")
            continue
    
        ok = executar_etapa(
            logger,
            f"CNES — {descricao}",
            geocnes,
            mun,
            uf,
            here_api,
            code_un=code_un
        )
    
        if not ok:
            etapas_com_falha.append(f"CNES tipo {code_un}")

    # --- Critérios C1–C9 ---
    etapas_criterios = [
        ("C1 — Vulnerabilidade Social",    "C1_VulnSoc",
         geosaude.vulnerabilidade,          (mun, uf, bbox)),
        ("C2/C3 — Demográfico e Renda",    "C2_DistDemog",
         geosaude.dados_demograficos,       (uf, mun, bbox)),
        ("C4/C5/C6 — Acessibilidade",      "C4_TempoMin",
         geosaude.travel_time_calculation,  (mun, uf, bbox, here_api, opentopo_api)),
        ("C7 — Risco de Eventos Naturais", "C7_EventNat",
         geosaude.sgb_data,                 (uf, mun, bbox)),
        ("C8 — Equipamentos Indesejáveis", "C8_EqupInd",
         geosaude.PUI,                      (mun, uf, bbox)),
        ("C9 — Equipamentos Desejáveis",   "C9_EqupDes",
         geosaude.equipamentos_desejaveis,  (mun, uf, bbox, here_api)),
    ]

    for nome, raster_saida, func, args in etapas_criterios:
        if not forcar_reprocessamento and criterio_ja_processado(mun, raster_saida):
            logger.info(f"PULADO (já existe em disco): {nome}")
            continue
        if not executar_etapa(logger, nome, func, *args):
            etapas_com_falha.append(nome)

    # --- Combinação ponderada ---
    if not executar_etapa(logger, "Calculadora Raster",
                          calculadora_raster, mun, uf):
        etapas_com_falha.append("Calculadora Raster")

    # --- Agregação H3 ---
    if not executar_etapa(logger, "Agregação H3",
                          agregar_resultados, mun, uf):
        etapas_com_falha.append("Agregação H3")

    # --- Locais prioritários ---
    if not executar_etapa(logger, "Top cells",
                          top_cells, mun, uf, here_api):
        etapas_com_falha.append("Top cells")

    # --- Relatório PDF ---
    if not executar_etapa(logger, "Geração do relatório PDF",
                          gerar_relatorio, mun, uf, "./data/logo_geosaude.png"):
        etapas_com_falha.append("Relatório PDF")
    
    if not executar_etapa(logger, "Geração do dashboard",
                          gerar_dashboard, mun, uf,"./data/logo_geosaude.png"):
        etapas_com_falha.append("Relatório PDF")

    # --- Resumo final ---
    logger.info(f"{'='*55}")
    logger.info(f"EXECUÇÃO FINALIZADA — {time.time() - t_total:.1f}s")

    if etapas_com_falha:
        logger.warning(f"{len(etapas_com_falha)} etapa(s) com falha:")
        for etapa in etapas_com_falha:
            logger.warning(f"  • {etapa}")
    else:
        logger.info("Todas as etapas concluídas com sucesso.")

    logger.info(f"{'='*55}")

In [3]:
import r5py
print(r5py.__version__)

1.1.3


In [4]:
# ============================================================
# CÉLULA DE EXECUÇÃO — preencha as variáveis e rode esta célula
# ============================================================

main(
    mun                  = "São Carlos",
    uf                   = "SP",
    here_api             = "arpqFeP0B5vkZfMy2v-hWNEIezuFJYrgSQiYb2OaVjY",
    opentopo_api         = "24f4d8988ffd5ce2a5ddac2499a1e0a0",
    forcar_reprocessamento = False  # True para reprocessar tudo
)

2026-04-06 08:27:50  INFO      GeoSaúde iniciado → município: São Carlos / UF: SP
2026-04-06 08:28:02  INFO      Bounding box: [-5353594.62866446 -2531007.30056049 -5312008.88769679 -2463159.80118995]
2026-04-06 08:28:02  INFO      Unidades de APS (code_un=02) já existe. Pulando etapa.
2026-04-06 08:28:02  INFO      Hospitais (code_un=05) já existe. Pulando etapa.
2026-04-06 08:28:02  INFO      UPAs (code_un=73) já existe. Pulando etapa.
2026-04-06 08:28:02  INFO      =======================================================
2026-04-06 08:28:02  INFO      INÍCIO: C1 — Vulnerabilidade Social
2026-04-06 08:28:02  INFO      =======================================================
✔ Dados de vulnerabilidade para São Carlos disponíveis no Índice Paulista de Vulnerabilidade Social (SEADE).
2026-04-06 08:28:04  INFO      CONCLUÍDO: C1 — Vulnerabilidade Social  (2.1s)
2026-04-06 08:28:04  INFO      =======================================================
2026-04-06 08:28:04  INFO      INÍCIO: C2/C

C:\Users\Lucas\anaconda3\envs\GeoSaude\Lib\site-packages\r5py\r5\base_travel_time_matrix.py:231: RuntimeWarning: Some origin points could not be snapped to the street network
  warnings.warn(


Executando análise de tempo mínimo...
Concluído.
Executando análise do nível de acesso...
Concluído.
Executando análise de cobertura...
Regiões únicas geradas: 252
Distribuição de scores:
score
1    94
2    59
3    57
4    39
Name: count, dtype: int64
C6 chamado!!
C6 gerado: 34 isócronas, 4 classes de cobertura.
Concluído.
2026-04-06 08:33:07  INFO      CONCLUÍDO: C4/C5/C6 — Acessibilidade  (267.1s)
2026-04-06 08:33:07  INFO      =======================================================
2026-04-06 08:33:07  INFO      INÍCIO: C7 — Risco de Eventos Naturais
2026-04-06 08:33:07  INFO      =======================================================
Pasta encontrada: ./temp/SP_São Carlos_Suscetibilidade/
✔ Dados de suscetibilidade encontrados:
   Pasta: ./temp/SP_São Carlos_Suscetibilidade/Suscetibilidade
   Shapefiles: 2
   GPKG: 0
Processando 2 shapefiles
Shapefile carregado: inundacao
Shapefile carregado: movimentodamassa
✔ Evento 'inundacao' reclassificado usando coluna 'CLASSE'.
✔ Evento 'mo